In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

dbutils.widgets.text("storageName", "adlsssmartdata2698")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("esquema", "bronze")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist_orders_dataset.csv"

In [0]:
df_orders = spark.read.option("header", True)\
                      .option("inferSchema", True)\
                      .csv(ruta)

In [0]:

orders_schema = StructType(fields=[
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])


In [0]:
df_orders_final = spark.read\
.option("header", True)\
.schema(orders_schema)\
.csv(ruta)


In [0]:
orders_selected_df = df_orders_final.select(
    col("order_id"),
    col("customer_id"),
    col("order_status"),
    col("order_purchase_timestamp"),
    col("order_approved_at"),
    col("order_delivered_carrier_date"),
    col("order_delivered_customer_date"),
    col("order_estimated_delivery_date")
)

In [0]:
orders_renamed_df = orders_selected_df

In [0]:
orders_final_df = orders_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
orders_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.orders_raw")